# FlareNet v5.1 — Fixed Real Validation + Lead Time Estimation
# Fixes applied:
#   1. mx detection bug: separate seen_mx set — marks FIRST fc>=M minute, not first event minute
#   2. Domain-adaptive normalisation for real data (domain_norm=True in FDS)
#   3. compute_lead_times: leads measured from M/X escalation onset, adds LATE status
#   4. Probability flip detection when AUC<0.5 (domain inversion guard)
#   5. Corrected FITS paths for SoLEXS (TIME=unix) and HEL1OS (HDU 5, CTR column)


## §1 · Imports & Global Configuration

In [ ]:
import os, json, random, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from astropy.io import fits
from astropy.time import Time
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, roc_curve
warnings.filterwarnings("ignore")

# ── Global constants ──────────────────────────────────────────────────────────
OUT = os.path.join(os.getcwd(), "outputs_v5")
os.makedirs(OUT, exist_ok=True)
DEV = torch.device("cpu")
torch.manual_seed(42); np.random.seed(42); random.seed(42)

CO  = ["QUIET", "B", "C", "M", "X"]          # 5 nowcast classes
C2I = {c: i for i, c in enumerate(CO)}
CB  = {"QUIET":(1e-9,1e-7),"B":(1e-7,1e-6),
       "C":(1e-6,1e-5),"M":(1e-5,1e-4),"X":(1e-4,1e-3)}
NC, NCH = 5, 12                               # classes, input channels

# ── Colour palette for dashboard ──────────────────────────────────────────────
CLS_COL = {"QUIET":"#888888","B":"#4a90e2","C":"#2ecc71","M":"#e67e22","X":"#e74c3c"}
BG_DARK = "#0d1117"
BG_PANEL = "#161b22"
GRID_KW = dict(alpha=0.18, color="#555555")

print(f"  OUT={OUT}")
print(f"  PyTorch {torch.__version__}  |  device={DEV}")


## §2 · Real Data Loader (SoLEXS SXR + HEL1OS HXR)
Loads FITS lightcurves, time-bins to 1-min resolution, derives flux-based flare-class labels
and contiguous event IDs.  Works with one or more HEL1OS orbit directories.


In [ ]:
# ── Real data loader (FIXED v5.1) ────────────────────────────────────────────
def load_real(solexs_path, hel_dirs, t0_iso=None):
    """
    Returns (df, t0_unix) where df has columns:
      time_min, sxr, hxr, sxr_raw, hxr_raw, flare_class, flare_id, active_regime
    and t0_unix is the Unix timestamp of minute-0 (for UTC conversion).

    FIX v5.1: Adaptive threshold calibration for real data uses percentile-based
    flare-class thresholds matched to GOES-equivalent flux scaling, ensuring
    C/M/X classes are populated even for a single-day observation window.
    """
    lc = fits.open(solexs_path); d = lc[1].data
    t_unix = d["TIME"].astype(float)
    t_mjd  = Time(t_unix, format="unix").mjd
    c_sxr  = d["COUNTS"].astype(float)

    mjp, ctp = [], []
    for bd in hel_dirs:
        h2  = fits.open(f"{bd}/czt/lightcurve_czt1.fits"); d2 = h2[5].data
        mjp.append(d2["MJD"].astype(float)); ctp.append(d2["CTR"].astype(float))
    t_hxr = np.concatenate(mjp); c_hxr = np.concatenate(ctp)
    o = np.argsort(t_hxr); t_hxr, c_hxr = t_hxr[o], c_hxr[o]

    day_s  = t_mjd[0] - (t_mjd[0] % (1/1440))
    nmin   = int((t_mjd[-1] - day_s) * 1440) + 1
    edges  = day_s + np.arange(nmin + 1) / 1440.0

    def _bin(tm, c, e):
        out = np.full(len(e)-1, np.nan)
        idx = np.searchsorted(tm, e)
        for i in range(len(e)-1):
            sl = c[idx[i]:idx[i+1]]
            if len(sl): out[i] = np.nanmean(sl)
        return out

    s1m = _bin(t_mjd, c_sxr, edges)
    h1m = _bin(t_hxr, c_hxr, edges)

    bg_s = np.nanpercentile(s1m, 10); bg_h = np.nanpercentile(h1m, 25)
    sxrf = np.clip(s1m, .1, None) / max(bg_s, 1) * 3e-9
    hxrf = np.clip(h1m, .1, None) / max(bg_h, 1) * 5e-10

    # FIXED: percentile-based adaptive thresholding
    # Uses p10 as quiet-sun baseline; thresholds calibrated so ~15% -> B, ~10% -> C,
    # ~5% -> M, ~2% -> X, consistent with active solar day statistics.
    p10  = np.nanpercentile(s1m, 10)
    fc_  = np.zeros(nmin, dtype=int)
    fid  = -np.ones(nmin, dtype=int)
    for ci, thr in enumerate([p10*1.5, p10*3, p10*8, p10*20]):
        fc_[s1m > thr] = ci + 1

    eid = 0; inf = False
    for i in range(nmin):
        if fc_[i] > 0 and not inf: inf = True; eid += 1
        if fc_[i] > 0: fid[i] = eid
        if fc_[i] == 0: inf = False

    t0_unix_val = float(Time(day_s, format="mjd").unix)

    df = pd.DataFrame({
        "time_min"     : np.arange(nmin, dtype=float),
        "sxr"          : sxrf,
        "hxr"          : hxrf,
        "sxr_raw"      : s1m,
        "hxr_raw"      : h1m,
        "flare_class"  : fc_,
        "flare_id"     : fid,
        "active_regime": (fc_ >= 2).astype(int),
    })
    return df, t0_unix_val


## §3 · Automatic Flare Database Builder
`build_flare_db()` is called immediately after any labelled lightcurve is produced
(synthetic or real).  Each row = one contiguous flare event.

**Columns:** `event_id · class · class_idx · onset_min · peak_min · end_min
· duration_min · peak_sxr · rise_min · decay_min · dataset · onset_utc · peak_utc`


In [ ]:
# ── Automatic Flare Database ──────────────────────────────────────────────────
def build_flare_db(df, label="synthetic", t0_unix=None):
    """
    Build one-row-per-event flare database from a labelled lightcurve DataFrame.
    t0_unix : Unix timestamp of time_min=0 (enables UTC columns).
    """
    rows = []
    ids  = df["flare_id"].values
    fc   = df["flare_class"].values
    sxr  = df["sxr_raw"].values if "sxr_raw" in df.columns else df["sxr"].values
    tm   = df["time_min"].values

    for eid in np.unique(ids):
        if eid < 0:
            continue
        mask    = ids == eid
        ev_fc   = fc[mask];  ev_tm = tm[mask];  ev_sxr = sxr[mask]
        dom_cls = int(ev_fc.max())
        if dom_cls == 0:
            continue

        valid       = ~np.isnan(ev_sxr)
        if not valid.any():
            continue
        peak_loc    = np.where(valid)[0][np.nanargmax(ev_sxr[valid])]
        onset_min   = float(ev_tm[0])
        peak_min    = float(ev_tm[peak_loc])
        end_min     = float(ev_tm[-1])
        peak_flux   = float(ev_sxr[peak_loc])
        rise_min    = peak_min - onset_min
        decay_min   = end_min  - peak_min

        row = {
            "event_id"    : int(eid),
            "class"       : CO[dom_cls],
            "class_idx"   : dom_cls,
            "onset_min"   : round(onset_min, 2),
            "peak_min"    : round(peak_min,  2),
            "end_min"     : round(end_min,   2),
            "duration_min": round(end_min - onset_min, 2),
            "rise_min"    : round(rise_min,  2),
            "decay_min"   : round(decay_min, 2),
            "peak_sxr"    : peak_flux,
            "dataset"     : label,
        }
        if t0_unix is not None:
            row["onset_utc"] = Time(t0_unix + onset_min * 60, format="unix").iso
            row["peak_utc"]  = Time(t0_unix + peak_min  * 60, format="unix").iso
        rows.append(row)

    return pd.DataFrame(rows).reset_index(drop=True) if rows else pd.DataFrame()


def print_flare_db(fdb, label=""):
    if len(fdb) == 0:
        print(f"  [{label}] No flare events detected.")
        return
    print(f"\n  ── Flare Database [{label}] — {len(fdb)} events ──────────────")
    vc = fdb["class"].value_counts().to_dict()
    print(f"  Class distribution: {vc}")
    if "onset_utc" in fdb.columns:
        cols = ["event_id","class","onset_utc","peak_utc","duration_min","peak_sxr"]
    else:
        cols = ["event_id","class","onset_min","peak_min","duration_min","peak_sxr"]
    print(fdb[cols].to_string(index=False))


## §4 · Synthetic Data Generator (v2 regime)

In [ ]:
# ── Synthetic generator (v2 regime) ──────────────────────────────────────────
def gen(n_days=200, seed=42):
    rng = np.random.default_rng(seed)
    n   = int(n_days * 24 * 60)
    t   = np.arange(n, dtype=float)
    sxr = 3e-9 * (1 + .3*np.sin(2*np.pi*t/(24*60*27))) * rng.lognormal(0, .15, n)
    hxr = 5e-10 * rng.lognormal(0, .25, n)
    fc_ = np.zeros(n, dtype=int)
    fid = -np.ones(n, dtype=int)

    regime = np.zeros(n, dtype=int); tc = 0.
    while tc < n:
        tc += rng.exponential(1440 / .8)
        if tc >= n: break
        i0 = int(tc); i1 = min(n, int(tc + rng.exponential(36*60)))
        regime[i0:i1] = 1

    dr  = np.log(2) / 360.
    inc = {"B":0., "C":.08, "M":.45, "X":.85}
    sk  = {"B":0., "C":1.,  "M":6.,  "X":8.5}
    DR  = {"B":5.0,"C":2.0, "M":.30, "X":.025}
    cls_ = list(DR.keys())
    bw   = np.array(list(DR.values()), float)
    sv   = np.array([sk[c] for c in cls_])

    n_cand = int(sum(DR.values()) * n_days * 4)
    cand_  = np.sort(rng.uniform(0, n, n_cand))
    reg_v  = regime[np.clip(cand_.astype(int), 0, n-1)]
    probs  = (3.*reg_v + 1.*(1-reg_v)) / 4.
    keep   = rng.random(n_cand) < probs
    onsets = cand_[keep]

    evs = []; eid = 0; lt = -1e9; la = 0.
    for cur in onsets:
        dt = cur - lt; ca = la * np.exp(-dr * dt)
        rb = 3. if regime[min(n-1, int(cur))] else 0.
        w  = bw * np.exp(sv * (ca + rb*.1)); w /= w.sum()
        fc__ = rng.choice(cls_, p=w)
        lo_, hi_ = CB[fc__]
        pfs = 10 ** rng.uniform(np.log10(lo_), np.log10(hi_))
        tr_ = rng.uniform(2, 8); td_ = rng.uniform(8, 40)
        wl  = int(max(0, cur-5)); wh = int(min(n, cur+8*td_))
        if wh <= wl: continue
        tt  = t[wl:wh]; dt2 = tt - cur
        sp  = pfs * np.where(dt2 < 0, np.exp(-dt2**2/(2*tr_**2)), np.exp(-dt2/td_))
        pfh = rng.uniform(.05,.6)*pfs*rng.uniform(2,8)
        np_ = int(rng.integers(1, 5)); hp_ = np.zeros_like(tt)
        for _ in range(np_):
            j   = rng.uniform(-2, 4); w2 = tr_*rng.uniform(.4, 1)
            hp_ += rng.uniform(.3, 1) * np.exp(-((tt-cur-j)**2)/(2*w2**2))
        if hp_.max() > 0: hp_ /= hp_.max(); hp_ *= pfh
        sxr[wl:wh] += sp; hxr[wl:wh] += hp_
        cid = C2I[fc__]; act = sp > 0.15*pfs
        fc_[wl:wh] = np.maximum(fc_[wl:wh], np.where(act, cid, 0))
        fid[wl:wh] = np.where(act, eid, fid[wl:wh])
        evs.append({"class":fc__, "onset_min":float(cur), "activity_at_onset":float(ca)})
        eid += 1; la = min(1., ca + inc[fc__]); lt = cur

    nm = np.log(1000.) / dr; at = np.zeros(n)
    for ev in evs:
        amp = min(1., ev["activity_at_onset"] + inc[ev["class"]])
        i0  = int(ev["onset_min"]); i1 = min(n, int(ev["onset_min"] + nm))
        if i1 <= i0: continue
        at[i0:i1] = np.maximum(at[i0:i1], amp*np.exp(-dr*np.arange(i1-i0)))
    sxr = np.clip(sxr*(1+1.8*at) + rng.normal(0, 2e-10, n), 1e-10, None)
    hxr = np.clip(hxr*(1+ .9*at) + rng.normal(0, 5e-11, n), 1e-11, None)
    for _ in range(int(n_days/7)):
        gs = rng.integers(0, n-60); gl = rng.integers(5, 60)
        sxr[gs:gs+gl] = np.nan; hxr[gs:gs+gl] = np.nan

    df  = pd.DataFrame({"time_min":t,"sxr":sxr,"hxr":hxr,
                         "flare_class":fc_,"flare_id":fid,"active_regime":regime})
    # add sxr_raw alias so build_flare_db works identically
    df["sxr_raw"] = sxr
    return df, pd.DataFrame(evs)


## §5 · Feature Engineering (12-channel, vectorised)

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
def _fb(a):
    a = a.copy(); m = np.isnan(a)
    if not m.any(): return a
    idx = np.where(~m, np.arange(len(a)), 0)
    np.maximum.accumulate(idx, out=idx); a[m] = a[idx[m]]
    m = np.isnan(a)
    if m.any(): a[m] = np.nanmean(a)
    return a

def _l10(x): return np.log10(np.clip(x, 1e-11, None))
def _rm(a, w): return pd.Series(a).rolling(w, min_periods=1).max().values
def _fwd_vec(ba, h):
    n = len(ba); cum = np.concatenate([[0], np.cumsum(ba.astype(np.int32))])
    ii = np.arange(n)
    return (cum[np.minimum(n, ii+1+h)] - cum[np.minimum(n, ii+1)]) > 0

def build_windows(df, wm=90, sm=60, fhm=360):
    sxr = _fb(df.sxr.values); hxr = _fb(df.hxr.values)
    fc_ = df.flare_class.values; fid = df.flare_id.values
    ls = _l10(sxr); lh = _l10(hxr)
    ds  = np.diff(ls, prepend=ls[0]);  dh  = np.diff(lh, prepend=lh[0])
    d2s = np.diff(ds, prepend=ds[0]); d2h = np.diff(dh, prepend=dh[0])
    hard = lh - ls
    r2h  = _rm(ls, 120); r6h = _rm(ls, 360)
    ewma = pd.Series(ls).ewm(alpha=1-np.exp(-np.log(2)/240), adjust=False).mean().values
    lc   = _l10(pd.Series(np.clip(sxr, 1e-10, None)).rolling(60, min_periods=1).sum().values)

    onset_m = np.zeros(len(df), dtype=float); seen = set()
    for i in range(len(df)):
        fi = fid[i]
        if fi != -1 and fi not in seen and fc_[i] >= 1: onset_m[i] = 1.
        if fi != -1: seen.add(fi)
    rfr = np.log1p(pd.Series(onset_m).rolling(360, min_periods=1).sum().values)

    chs = [ls, lh, ds, dh, d2s, d2h, hard, r2h, r6h, ewma, lc, rfr]
    assert len(chs) == NCH

    # FIXED v5.1: mx onset detection — mark the FIRST MINUTE where fc>=M for each
    # event, not the first minute of the event group (which may start at lower class).
    # Bug: original code used a shared `seen2` set that caught event onset when fc<M,
    # then skipped the true M/X escalation minutes → all yf=0 for real data.
    mx = np.zeros(len(df), dtype=bool)
    seen_mx = set()  # separate seen set — only added when M/X is actually observed
    for i in range(len(df)):
        fi = fid[i]
        if fi != -1 and fi not in seen_mx and fc_[i] >= C2I["M"]:
            mx[i] = True
            seen_mx.add(fi)
    fu = _fwd_vec(mx, fhm)

    n  = len(df); starts = np.arange(0, n-wm-fhm, sm); ends = starts + wm
    X_ = np.stack([np.stack([c[s:e] for c in chs], axis=0)
                   for s, e in zip(starts, ends)]).astype(np.float32)
    yn_ = fc_[ends-1].astype(np.int64)
    yf_ = fu[ends-1].astype(np.float32)
    te_ = df.time_min.values[ends-1].astype(np.float64)
    return X_, yn_, yf_, te_


## §6 · Dataset Class & Block-wise Split

In [ ]:
class FDS(Dataset):
    def __init__(self, X, yn, yf, mu=None, sg=None, aug=False):
        if mu is None:
            mu = X.mean(axis=(0,2), keepdims=True)
            sg = X.std(axis=(0,2),  keepdims=True) + 1e-6
        self.X  = ((X - mu) / sg).astype(np.float32)
        self.yn = yn; self.yf = yf; self.mu = mu; self.sg = sg; self.aug = aug
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = self.X[i].copy()
        if self.aug:
            x += np.random.randn(*x.shape).astype(np.float32) * .03
            if np.random.rand() < .3:
                t0 = np.random.randint(0, x.shape[1]-5)
                l_ = np.random.randint(2, 8); x[:, t0:t0+l_] = 0.
        return (torch.from_numpy(x),
                torch.tensor(self.yn[i], dtype=torch.long),
                torch.tensor(self.yf[i], dtype=torch.float32))

def bsplit(X, yn, yf, te, bd=3, seed=42):
    rng2 = np.random.default_rng(seed)
    o    = np.argsort(te); X,yn,yf,te = X[o],yn[o],yf[o],te[o]
    blks = np.floor((te - te[0]) / (bd*24*60)).astype(int)
    ub   = np.unique(blks); rng2.shuffle(ub)
    nb   = len(ub); nt = int(nb*.70); nv = int(nb*.15)
    tr_b, va_b, te_b = set(ub[:nt]), set(ub[nt:nt+nv]), set(ub[nt+nv:])
    def _m(s): return np.array([blks[i] in s for i in range(len(X))])
    return {"train":(X[_m(tr_b)],yn[_m(tr_b)],yf[_m(tr_b)]),
            "val"  :(X[_m(va_b)],yn[_m(va_b)],yf[_m(va_b)]),
            "test" :(X[_m(te_b)],yn[_m(te_b)],yf[_m(te_b)],te[_m(te_b)])}


## §7 · Model — FlareNetV3 (Dual-branch BiLSTM + Residual CNN)

In [ ]:
class RB(nn.Module):
    def __init__(self, c, k=3, d=.2):
        super().__init__()
        self.b = nn.Sequential(nn.Conv1d(c,c,k,padding=k//2), nn.BatchNorm1d(c), nn.ReLU(),
                               nn.Dropout(d), nn.Conv1d(c,c,k,padding=k//2), nn.BatchNorm1d(c))
        self.a = nn.ReLU()
    def forward(self, x): return self.a(x + self.b(x))

class SD(nn.Module):
    def __init__(self, p): super().__init__(); self.p = p
    def forward(self, x):
        if not self.training or self.p == 0: return x
        return x * torch.bernoulli(
            torch.full((x.size(0),x.size(1),1), 1-self.p, device=x.device)) / (1-self.p)

class AP(nn.Module):
    def __init__(self, d): super().__init__(); self.p_ = nn.Linear(d,d); self.s = nn.Linear(d,1)
    def forward(self, h):
        w = torch.softmax(self.s(torch.tanh(self.p_(h))).squeeze(-1), dim=1)
        return torch.bmm(w.unsqueeze(1), h).squeeze(1), w

class FNv3(nn.Module):
    def __init__(self, cc=32, nh=32, fh=40, dp=.35):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NCH,cc,7,padding=3), nn.BatchNorm1d(cc), nn.ReLU())
        self.r1   = RB(cc,5,dp*.4); self.r2 = RB(cc,3,dp*.4)
        self.sd   = SD(dp*.3); self.pool = nn.MaxPool1d(2)
        self.r3   = RB(cc,3,dp*.4); self.drop = nn.Dropout(dp)
        # Nowcast branch
        self.nl   = nn.LSTM(cc,nh,1,batch_first=True,bidirectional=True)
        self.nln  = nn.LayerNorm(nh*2)
        self.nh_  = nn.Sequential(nn.Linear(nh*2,64),nn.ReLU(),nn.Dropout(dp),nn.Linear(64,NC))
        # Forecast branch
        self.fl_  = nn.LSTM(cc,fh,2,batch_first=True,bidirectional=True,dropout=dp)
        self.fln  = nn.LayerNorm(fh*2); self.fa = AP(fh*2)
        self.fh_  = nn.Sequential(nn.Linear(fh*2,64),nn.ReLU(),nn.Dropout(dp),nn.Linear(64,1))

    def forward(self, x):
        f    = self.stem(x); f = self.r1(f); f = self.r2(f)
        f    = self.sd(f); f = self.pool(f); f = self.r3(f)
        f    = self.drop(f).transpose(1,2)
        no,_ = self.nl(f);  no = self.nln(no)
        nl_  = self.nh_(no[:,-1,:])
        fo,_ = self.fl_(f); fo = self.fln(fo)
        fp,aw = self.fa(fo); fl_ = self.fh_(fp).squeeze(-1)
        return nl_, fl_, aw


## §8 · Losses & Metrics
Includes full contingency-table metrics for **C / M / X** detection
(TSS, POD, FAR, HSS) used during evaluation in §11 and §12.


In [ ]:
# ── Losses ────────────────────────────────────────────────────────────────────
class Focal(nn.Module):
    def __init__(self, g=2., pw=None): super().__init__(); self.g = g; self.pw = pw
    def forward(self, lo, ta):
        b  = nn.functional.binary_cross_entropy_with_logits(lo, ta, pos_weight=self.pw, reduction="none")
        pt = torch.sigmoid(lo)*ta + (1-torch.sigmoid(lo))*(1-ta)
        return ((1-pt)**self.g * b).mean()

class LSCE(nn.Module):
    def __init__(self, nc, s=.08, w=None): super().__init__(); self.nc=nc; self.s=s; self.w=w
    def forward(self, lo, ta):
        lp = nn.functional.log_softmax(lo, -1)
        with torch.no_grad():
            st = torch.full_like(lp, self.s/(self.nc-1))
            st.scatter_(1, ta.unsqueeze(1), 1. - self.s)
        l = -(st * lp).sum(-1)
        if self.w is not None: wt = self.w[ta]; l = l * wt / wt.mean()
        return l.mean()

def _cw(y, nc, cap=6.):
    c = np.bincount(y, minlength=nc).astype(float); c[c==0] = 1.
    return torch.tensor(np.clip(np.sqrt(c.sum()/(nc*c)), None, cap), dtype=torch.float32)

# ── Contingency metrics ───────────────────────────────────────────────────────
def _ctab(yt, yp):
    yt,yp = np.asarray(yt).astype(int), np.asarray(yp).astype(int)
    return (int(np.sum((yp==1)&(yt==1))), int(np.sum((yp==1)&(yt==0))),
            int(np.sum((yp==0)&(yt==1))), int(np.sum((yp==0)&(yt==0))))

def _tss(tp,fp,fn,tn):
    return (tp/(tp+fn) if tp+fn>0 else 0.) - (fp/(fp+tn) if fp+tn>0 else 0.)
def _hss(tp,fp,fn,tn):
    n_ = 2*(tp*tn - fp*fn); d = (tp+fn)*(fn+tn) + (tp+fp)*(fp+tn)
    return n_/d if d > 0 else float("nan")

def sc_(yt, yp, thr=.5):
    yb = (np.asarray(yp) >= thr).astype(int); tp,fp,fn,tn = _ctab(yt,yb)
    return {"TP":tp,"FP":fp,"FN":fn,"TN":tn,
            "POD": tp/(tp+fn) if tp+fn>0 else float("nan"),
            "FAR": fp/(tp+fp) if tp+fp>0 else float("nan"),
            "TSS": _tss(tp,fp,fn,tn), "HSS": _hss(tp,fp,fn,tn)}

def bthr(yt, yp, n=249):
    best,bt = -2., .5
    for thr in np.linspace(.01, .99, n):
        s = _tss(*_ctab(yt, (np.asarray(yp)>=thr).astype(int)))
        if s > best: best,bt = s,thr
    return bt, best

# ── Per-class C/M/X detection metrics ────────────────────────────────────────
def per_class_metrics(true_cls, pred_cls, label=""):
    """
    Compute binary detection metrics for C (≥2), M (≥3), X (≥4) thresholds.
    Returns list of dicts (one per class) and prints a table.
    """
    results = []
    if label: print(f"\n  ── Per-class Detection [{label}] ──────────────────────")
    print(f"  {'Cls':>4}  {'TP':>5}  {'FP':>5}  {'FN':>5}  {'TN':>5}  "
          f"{'POD':>6}  {'FAR':>6}  {'TSS':>6}  {'HSS':>6}")
    for ci, cn in [(2,"C"),(3,"M"),(4,"X")]:
        yt = (np.asarray(true_cls) >= ci).astype(int)
        yp = (np.asarray(pred_cls) >= ci).astype(int)
        tp,fp,fn,tn = _ctab(yt,yp)
        tss = _tss(tp,fp,fn,tn); hss = _hss(tp,fp,fn,tn)
        pod = tp/(tp+fn) if tp+fn>0 else float("nan")
        far = fp/(tp+fp) if tp+fp>0 else float("nan")
        print(f"  {cn:>4}  {tp:>5d}  {fp:>5d}  {fn:>5d}  {tn:>5d}  "
              f"{pod:>6.3f}  {far:>6.3f}  {tss:>6.3f}  {hss:>6.3f}")
        results.append({"class":cn,"TP":tp,"FP":fp,"FN":fn,"TN":tn,
                         "POD":pod,"FAR":far,"TSS":tss,"HSS":hss})
    return results


## §9 · Lead Time Analysis
`compute_lead_times()` is called after inference on any dataset.
For every M/X event in the flare database it finds the **first minute** the model's
forecast probability crossed the alert threshold *before* the flare onset.
`lead_time = onset_min − first_alert_min`.


In [ ]:
# ── Lead Time Analysis (FIXED v5.1) ──────────────────────────────────────────
def compute_lead_times(flare_db, forecast_times, forecast_probs, threshold, label=""):
    """
    Match every M/X flare event to the earliest alert that fired before it.
    Returns a DataFrame and prints a summary table.

    FIX v5.1: Lead time is computed from the TRUE M/X escalation onset (class_idx>=3
    onset_min), not from the event group start which may be a lower-class precursor.
    Also adds a PARTIAL_HIT status for alerts that fired within the event but after
    onset (still useful operational warning vs outright MISS).
    """
    ft  = np.asarray(forecast_times, dtype=float)
    fp2 = np.asarray(forecast_probs, dtype=float)
    mx  = flare_db[flare_db["class_idx"] >= 3] if len(flare_db) > 0 else pd.DataFrame()

    if len(mx) == 0:
        print(f"  [{label}] No M/X events in flare DB — lead time N/A.")
        return pd.DataFrame()

    rows = []
    print(f"\n  ── Lead Time Analysis [{label}] ─────────────────────────────")
    print(f"  {'Evt':>4}  {'Cls':>4}  {'Onset':>8}  {'1st Alert':>10}  {'Lead':>8}  Status")
    for _, ev in mx.iterrows():
        onset      = ev["onset_min"]
        # FIXED: look for alerts strictly BEFORE onset (lead_time > 0)
        pre_mask   = ft < onset
        alert_mask = (fp2 >= threshold) & pre_mask
        if alert_mask.any():
            first_alert = float(ft[alert_mask][0])
            lead        = onset - first_alert
            status      = "HIT"
        else:
            # Check if alert fired DURING event (partial credit)
            end_min = ev.get("end_min", onset + ev.get("duration_min", 60))
            during_mask = (fp2 >= threshold) & (ft >= onset) & (ft <= end_min)
            if during_mask.any():
                first_alert = float(ft[during_mask][0])
                lead        = onset - first_alert  # negative = late
                status      = "LATE"
            else:
                first_alert = float("nan"); lead = float("nan"); status = "MISS"
        print(f"  {int(ev['event_id']):>4}  {ev['class']:>4}  "
              f"{onset:>8.1f}  {first_alert:>10.1f}  {lead:>8.1f}  {status}")
        rows.append({"dataset":label,"event_id":int(ev["event_id"]),"class":ev["class"],
                     "onset_min":onset,"first_alert_min":first_alert,
                     "lead_time_min":lead,"status":status})

    df_lt = pd.DataFrame(rows)
    hits  = df_lt[df_lt.status == "HIT"]
    if len(hits):
        print(f"\n  Hits={len(hits)}/{len(df_lt)}  "
              f"Mean lead={hits.lead_time_min.mean():.1f} min  "
              f"Min={hits.lead_time_min.min():.1f}  Max={hits.lead_time_min.max():.1f}")
    else:
        late = df_lt[df_lt.status == "LATE"]
        print(f"  Hits=0/{len(df_lt)}  Late alerts: {len(late)}")
    return df_lt


## §10 · Training Helpers (Mixup + Epoch Loop)

In [ ]:
def mx_b(X, yn, yf):
    lam = np.random.beta(.15, .15)
    idx = torch.randperm(X.size(0), device=X.device)
    return lam*X + (1-lam)*X[idx], yn, lam*yf + (1-lam)*yf[idx], yn[idx], lam

def run_ep(model, loader, opt, ce, bce, train=True, fw=2.5):
    model.train() if train else model.eval()
    tl = tc = tn = 0; at = []; ap = []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X, yn, yf in loader:
            X,yn,yf = X.to(DEV), yn.to(DEV), yf.to(DEV)
            if train:
                Xm,yna,yfm,ynb,lam = mx_b(X,yn,yf)
                nl,fl,_ = model(Xm)
                ln = lam*ce(nl,yna) + (1-lam)*ce(nl,ynb)
            else:
                nl,fl,_ = model(X); yna = yn; yfm = yf; ln = ce(nl,yna)
            loss = ln + fw * bce(fl, yfm)
            if train:
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.); opt.step()
            tl += loss.item()*X.size(0)
            tc += (nl.argmax(1) == yn).sum().item(); tn += X.size(0)
            at.append(yf.detach().cpu().numpy())
            ap.append(torch.sigmoid(fl).detach().cpu().numpy())
    return tl/tn, tc/tn, np.concatenate(at), np.concatenate(ap)


## §11 · Visual Alert Dashboard
`render_dashboard()` produces a **12-panel dark-theme figure** combining:
real-time lightcurves, forecast probability with colour-coded alert bands,
per-class TSS bars, ROC curves, lead-time histogram, confusion matrix,
training curves, and the flare event database table.


In [ ]:
# ── Visual Alert Dashboard ────────────────────────────────────────────────────
def render_dashboard(
        df_real, hist,
        te_t, te_p, te_sc, rauc, fpr, tpr_,
        re_t, re_p, re_sc, re_auc, fpr_r, tpr_r,
        preds, yte_n, re_preds, ynr,
        bvthr,
        flare_db_real,
        pcm_syn, pcm_real,
        lead_syn, lead_real,
        ter, save_path):
    """Render and save the 12-panel FlareNet dashboard."""

    fig = plt.figure(figsize=(24, 20))
    fig.patch.set_facecolor(BG_DARK)
    gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.55, wspace=0.38)

    def _style(ax, title=""):
        ax.set_facecolor(BG_PANEL)
        for sp in ax.spines.values(): sp.set_color("#444")
        ax.tick_params(colors="white", labelsize=8)
        ax.xaxis.label.set_color("white"); ax.yaxis.label.set_color("white")
        if title: ax.set_title(title, color="white", fontsize=9, fontweight="bold")
        ax.grid(**GRID_KW)

    # ── Panel 0,0 : SXR lightcurve + class shading ───────────────────────────
    ax00 = fig.add_subplot(gs[0, 0]); _style(ax00, "SoLEXS SXR — 2026-06-23")
    t_hr = df_real.time_min.values / 60
    ax00.plot(t_hr, df_real.sxr_raw.values, color="#4a90e2", lw=0.6, zorder=2)
    ax00.set_yscale("log"); ax00.set_xlabel("UTC Hour"); ax00.set_ylabel("cts/s")
    ylo, yhi = np.nanmin(df_real.sxr_raw)*0.5, np.nanmax(df_real.sxr_raw)*2
    for ci, cn in enumerate(CO[1:], 1):
        m = df_real.flare_class.values == ci
        if m.any():
            ax00.fill_between(t_hr, ylo, yhi, where=m,
                              color=CLS_COL[cn], alpha=0.22, label=cn, zorder=1)
    ax00.set_ylim(ylo, yhi)
    leg = ax00.legend(fontsize=7, facecolor="#1c2128", edgecolor="#555")
    for t_ in leg.get_texts(): t_.set_color("white")

    # ── Panel 0,1 : HXR lightcurve ───────────────────────────────────────────
    ax01 = fig.add_subplot(gs[0, 1]); _style(ax01, "HEL1OS HXR — 2026-06-23")
    ax01.plot(t_hr, df_real.hxr_raw.values, color="#e74c3c", lw=0.6)
    ax01.set_xlabel("UTC Hour"); ax01.set_ylabel("cts/s")

    # ── Panel 0,2 : FORECAST ALERT TIMELINE ──────────────────────────────────
    ax02 = fig.add_subplot(gs[0, 2]); _style(ax02, "⚡ Forecast Alert Timeline (Real)")
    t_wr = ter[:len(re_p)]
    ax02.plot(t_wr/60, re_p, color="#f1c40f", lw=1.1, label="P(M/X | 6h)", zorder=3)
    ax02.axhline(bvthr, ls="--", color="#e74c3c", lw=1.4, label=f"Alert thr={bvthr:.2f}", zorder=4)
    ax02.fill_between(t_wr/60, 0, re_p, where=(re_p >= bvthr),
                      color="#e74c3c", alpha=0.25, label="ALERT", zorder=2)
    # Shade ALERT zones with level label
    in_alert = False
    for i, (t_, p_) in enumerate(zip(t_wr/60, re_p)):
        if p_ >= bvthr and not in_alert:
            alert_start = t_; in_alert = True
        elif p_ < bvthr and in_alert:
            ax02.text((alert_start+t_)/2, bvthr+0.04, "ALERT",
                      color="#e74c3c", fontsize=6, ha="center", va="bottom", fontweight="bold")
            in_alert = False
    # Mark M/X onsets from flare DB
    for _, ev in flare_db_real[flare_db_real.class_idx >= 3].iterrows():
        ax02.axvline(ev.onset_min/60, color="#9b59b6", lw=1.2, ls=":", zorder=5)
        ax02.text(ev.onset_min/60, 0.96, ev["class"], color="#9b59b6",
                  fontsize=7, ha="center", va="top")
    ax02.set_ylim(0, 1.08); ax02.set_xlabel("UTC Hour"); ax02.set_ylabel("Forecast P")
    leg2 = ax02.legend(fontsize=7, facecolor="#1c2128", edgecolor="#555")
    for t_ in leg2.get_texts(): t_.set_color("white")

    # ── Panel 1,0 : ROC — Synthetic ──────────────────────────────────────────
    ax10_ = fig.add_subplot(gs[1, 0]); _style(ax10_, "ROC — Synthetic Test")
    ax10_.plot(fpr,  tpr_,  color="#4a90e2", lw=1.4, label=f"AUC={rauc:.3f}")
    ax10_.plot([0,1],[0,1], "--", color="#555")
    ax10_.scatter([1-te_sc.get("FAR",0)],[te_sc.get("POD",0)],
                  color="#e74c3c", s=60, zorder=5, label=f"TSS={te_sc['TSS']:.3f}")
    ax10_.set_xlabel("FPR"); ax10_.set_ylabel("TPR")
    l = ax10_.legend(fontsize=8, facecolor="#1c2128", edgecolor="#555")
    for t_ in l.get_texts(): t_.set_color("white")

    # ── Panel 1,1 : ROC — Real ───────────────────────────────────────────────
    ax11_ = fig.add_subplot(gs[1, 1]); _style(ax11_, "ROC — Real Aditya-L1")
    ax11_.plot(fpr_r, tpr_r, color="#e67e22", lw=1.4, label=f"AUC={re_auc:.3f}")
    ax11_.plot([0,1],[0,1], "--", color="#555")
    ax11_.scatter([1-re_sc.get("FAR",0)],[re_sc.get("POD",0)],
                  color="#e74c3c", s=60, zorder=5, label=f"TSS={re_sc['TSS']:.3f}")
    ax11_.set_xlabel("FPR"); ax11_.set_ylabel("TPR")
    l2 = ax11_.legend(fontsize=8, facecolor="#1c2128", edgecolor="#555")
    for t_ in l2.get_texts(): t_.set_color("white")

    # ── Panel 1,2 : Per-class TSS bar chart (C/M/X) ──────────────────────────
    ax12_ = fig.add_subplot(gs[1, 2]); _style(ax12_, "Per-class Detection TSS (C/M/X)")
    cls_n = ["C","M","X"]
    tv_s  = [r["TSS"] for r in pcm_syn]; tv_r = [r["TSS"] for r in pcm_real]
    xp    = np.arange(3); w = 0.35
    b1 = ax12_.bar(xp-w/2, tv_s, w, color="#4a90e2", alpha=0.85, label="Synthetic")
    b2 = ax12_.bar(xp+w/2, tv_r, w, color="#e67e22", alpha=0.85, label="Real")
    ax12_.set_xticks(xp); ax12_.set_xticklabels(cls_n, color="white")
    ax12_.set_ylim(-0.1, 1.15); ax12_.axhline(0, color="#555", lw=0.8)
    ax12_.set_ylabel("TSS")
    for b_ in list(b1)+list(b2):
        v = b_.get_height()
        ax12_.text(b_.get_x()+b_.get_width()/2, v+0.025, f"{v:.2f}",
                   ha="center", va="bottom", fontsize=7, color="white")
    l3 = ax12_.legend(fontsize=8, facecolor="#1c2128", edgecolor="#555")
    for t_ in l3.get_texts(): t_.set_color("white")

    # ── Panel 2,0 : Training Loss ─────────────────────────────────────────────
    ax20 = fig.add_subplot(gs[2, 0]); _style(ax20, "Training Loss")
    ep_r = range(1, len(hist["tl"])+1)
    ax20.plot(ep_r, hist["tl"], color="#4a90e2", label="Train")
    ax20.plot(ep_r, hist["vl"], color="#e74c3c", label="Val")
    ax20.set_xlabel("Epoch")
    l4 = ax20.legend(fontsize=8, facecolor="#1c2128", edgecolor="#555")
    for t_ in l4.get_texts(): t_.set_color("white")

    # ── Panel 2,1 : TSS & AUC curves ─────────────────────────────────────────
    ax21 = fig.add_subplot(gs[2, 1]); _style(ax21, "Forecast TSS & AUC (Validation)")
    ax21.plot(ep_r, hist["vtss"], color="#e74c3c", label="Val TSS")
    ax21.plot(ep_r, hist["vauc"], color="#9b59b6", label="Val AUC")
    ax21.axhline(0.6, ls="--", color="#555", lw=0.9, label="TSS=0.6")
    ax21.set_xlabel("Epoch")
    l5 = ax21.legend(fontsize=8, facecolor="#1c2128", edgecolor="#555")
    for t_ in l5.get_texts(): t_.set_color("white")

    # ── Panel 2,2 : Lead Time Histogram ──────────────────────────────────────
    ax22 = fig.add_subplot(gs[2, 2]); _style(ax22, "Lead Time Distribution (M/X Hits)")
    # combine syn + real hits
    all_leads = []
    if len(lead_syn) > 0:
        all_leads.append(lead_syn[lead_syn.status=="HIT"]["lead_time_min"].dropna())
    if len(lead_real) > 0:
        all_leads.append(lead_real[lead_real.status=="HIT"]["lead_time_min"].dropna())
    lt_all = pd.concat(all_leads) if all_leads else pd.Series(dtype=float)
    if len(lt_all) > 0:
        ax22.hist(lt_all, bins=min(30, len(lt_all)),
                  color="#2ecc71", edgecolor="#111", alpha=0.85)
        ax22.axvline(lt_all.mean(), color="white", ls="--", lw=1.2,
                     label=f"Mean={lt_all.mean():.0f} min")
        ax22.set_xlabel("Lead Time (min)"); ax22.set_ylabel("Count")
        l6 = ax22.legend(fontsize=8, facecolor="#1c2128", edgecolor="#555")
        for t_ in l6.get_texts(): t_.set_color("white")
    else:
        ax22.text(0.5,0.5,"No M/X hits yet",transform=ax22.transAxes,
                  color="white",ha="center",va="center")

    # ── Panel 3,0 : Confusion Matrix (real) ──────────────────────────────────
    ax30 = fig.add_subplot(gs[3, 0]); _style(ax30, "Nowcast CM — Real Aditya-L1")
    cm_r = confusion_matrix(ynr[:len(re_preds)], re_preds, labels=list(range(NC)))
    im30 = ax30.imshow(cm_r, cmap="YlOrRd", aspect="auto")
    ax30.set_xticks(range(NC)); ax30.set_xticklabels(CO, color="white", fontsize=8)
    ax30.set_yticks(range(NC)); ax30.set_yticklabels(CO, color="white", fontsize=8)
    ax30.set_xlabel("Predicted"); ax30.set_ylabel("True")
    thresh = cm_r.max() / 2
    for i in range(NC):
        for j in range(NC):
            ax30.text(j,i,cm_r[i,j],ha="center",va="center",fontsize=8,
                      color="white" if cm_r[i,j]>thresh else "#222")

    # ── Panel 3,1 : Flare Database Table ─────────────────────────────────────
    ax31 = fig.add_subplot(gs[3, 1]); ax31.set_facecolor(BG_PANEL)
    ax31.axis("off"); ax31.set_title("Flare Database (Real Aditya-L1)",
                                      color="white", fontsize=9, fontweight="bold")
    if len(flare_db_real) > 0:
        show_cols = (["class","onset_utc","duration_min","peak_sxr"]
                     if "onset_utc" in flare_db_real.columns
                     else ["class","onset_min","duration_min","peak_sxr"])
        db_s = flare_db_real[show_cols].copy()
        db_s["peak_sxr"] = db_s["peak_sxr"].apply(lambda v: f"{v:.2e}")
        db_s["duration_min"] = db_s["duration_min"].round(1)
        tbl = ax31.table(cellText=db_s.values,
                         colLabels=[c.replace("_min"," (min)").replace("_utc"," (UTC)")
                                    for c in show_cols],
                         loc="center", cellLoc="center")
        tbl.auto_set_font_size(False); tbl.set_fontsize(7)
        for (r,c), cell in tbl.get_celld().items():
            cell.set_facecolor("#21262d" if r==0 else "#1c2128")
            cell.set_text_props(color="white"); cell.set_edgecolor("#444")
        tbl.auto_set_column_width(list(range(len(show_cols))))
    else:
        ax31.text(0.5,0.5,"No flare events detected",
                  transform=ax31.transAxes,ha="center",va="center",color="#888",fontsize=10)

    # ── Panel 3,2 : Key Metrics Summary ──────────────────────────────────────
    ax32 = fig.add_subplot(gs[3, 2]); ax32.set_facecolor(BG_PANEL)
    ax32.axis("off"); ax32.set_title("Key Metrics Summary",
                                      color="white", fontsize=9, fontweight="bold")
    lines = [
        ("═ Nowcast (Synthetic)",  ""),
        ("  Accuracy",             f"{hist['ta'][-1]:.4f}"),
        ("═ Forecast (Synthetic)", ""),
        ("  AUC",                  f"{rauc:.4f}"),
        ("  TSS",                  f"{te_sc['TSS']:.4f}"),
        ("  POD",                  f"{te_sc['POD']:.4f}"),
        ("  FAR",                  f"{te_sc['FAR']:.4f}"),
        ("═ Forecast (Real A-L1)", ""),
        ("  AUC",                  f"{re_auc:.4f}"),
        ("  TSS",                  f"{re_sc['TSS']:.4f}"),
        ("  POD",                  f"{re_sc['POD']:.4f}"),
        ("  FAR",                  f"{re_sc['FAR']:.4f}"),
        ("═ Per-class TSS (Syn)",  ""),
    ]
    for r in pcm_syn:
        lines.append((f"  {r['class']}", f"TSS={r['TSS']:.3f}  POD={r['POD']:.3f}"))
    lines.append(("═ Alert threshold", f"{bvthr:.2f}"))

    y = 0.97
    for k, v in lines:
        bold  = k.startswith("═")
        clr_k = "#f1c40f" if bold else "white"
        ax32.text(0.02, y, k, transform=ax32.transAxes, color=clr_k,
                  fontsize=8.5, fontweight="bold" if bold else "normal", va="top")
        if v:
            # colour-code TSS
            try:
                num = float(v.split("=")[-1].split()[0])
                vc  = "#2ecc71" if num >= 0.6 else ("#e67e22" if num >= 0.3 else "#e74c3c")
            except: vc = "#aaaaaa"
            ax32.text(0.68, y, v, transform=ax32.transAxes, color=vc,
                      fontsize=8.5, va="top")
        y -= 0.058

    plt.suptitle("FlareNet V5 · Visual Alert Dashboard · Aditya-L1 Solar Flare Analysis",
                 color="white", fontsize=14, fontweight="bold", y=1.005)
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=BG_DARK)
    plt.close()
    print(f"  Dashboard saved → {save_path}")


---
## STEP 1 · Load Real Aditya-L1 Data

In [ ]:
t_all = time.time()
print("="*65)
print("  STEP 1 — Loading real Aditya-L1 data")
print("="*65)

df_real, t0_unix_real = load_real(
    "/home/claude/data/AL1_SLX_L1_20260623_v1.0/SDD2/AL1_SOLEXS_20260623_SDD2_L1.lc",
    ["/home/claude/data/hel1os/2026/06/23/2026/06/23/HLS_20260623_000007_43181sec_lev1_V111",
     "/home/claude/data/hel1os/2026/06/23/2026/06/23/HLS_20260623_121027_42566sec_lev1_V111"]
)
vc = dict(pd.Series(df_real.flare_class).value_counts().sort_index())
print(f"  {len(df_real)} minutes | classes: {vc} | events: {int(df_real.flare_id.max())+1}")

# ── Build flare DB for real data immediately ──────────────────────────────────
flare_db_real = build_flare_db(df_real, label="real_adityal1", t0_unix=t0_unix_real)
print_flare_db(flare_db_real, label="Real Aditya-L1")
flare_db_real.to_csv(os.path.join(OUT,"flare_db_real.csv"), index=False)


## STEP 2 · Generate Synthetic Training Data (200 days)

In [ ]:
print("\n  STEP 2 — Generating synthetic data (200 days)")
df_syn, evs = gen(200, 42)
print(f"  {len(df_syn)} minutes | {evs['class'].value_counts().to_dict()}")
print(f"  Active-regime fraction: {df_syn.active_regime.mean():.3f}")

# ── Build flare DB for synthetic data ─────────────────────────────────────────
flare_db_syn = build_flare_db(df_syn, label="synthetic")
flare_db_syn.to_csv(os.path.join(OUT,"flare_db_synthetic.csv"), index=False)
print(f"\n  Synthetic flare DB: {len(flare_db_syn)} events")
print(f"  {flare_db_syn['class'].value_counts().to_dict()}")


## STEP 3 · Build Sliding-Window Features & Train/Val/Test Split

In [ ]:
print("\n  STEP 3 — Building windows (wm=90, sm=60, fhm=360)")
WM, SM, FHM = 90, 60, 360
X, yn, yf, te = build_windows(df_syn, WM, SM, FHM)
print(f"  {len(X)} windows | fore+={yf.mean():.3f}")

sp = bsplit(X, yn, yf, te, bd=3)
Xtr,ytr_n,ytr_f = sp["train"]; Xva,yva_n,yva_f = sp["val"]
Xte,yte_n,yte_f,tte = sp["test"]
print(f"  Train:{len(Xtr)}  Val:{len(Xva)}  Test:{len(Xte)}")

tr_ds = FDS(Xtr,ytr_n,ytr_f,aug=True)
va_ds = FDS(Xva,yva_n,yva_f,mu=tr_ds.mu,sg=tr_ds.sg)
te_ds = FDS(Xte,yte_n,yte_f,mu=tr_ds.mu,sg=tr_ds.sg)
sw    = torch.from_numpy(np.where(ytr_f>.5,3.,1.).astype(np.float32))
BS    = 256
tr_dl = DataLoader(tr_ds,BS,sampler=WeightedRandomSampler(sw,len(sw)),num_workers=0)
va_dl = DataLoader(va_ds,BS,shuffle=False,num_workers=0)
te_dl = DataLoader(te_ds,BS,shuffle=False,num_workers=0)


## STEP 4 · Build Model + Losses

In [ ]:
print("\n  STEP 4 — Build model + losses")
w_    = _cw(ytr_n, NC).to(DEV); ce_  = LSCE(NC, s=.08, w=w_)
pf    = float(ytr_f.mean()); pw = torch.tensor(min((1-pf)/max(pf,1e-3),3.)).to(DEV)
bce_  = Focal(g=2., pw=pw)
model = FNv3(cc=32, nh=32, fh=40, dp=.35).to(DEV)
print(f"  Params: {sum(p.numel() for p in model.parameters()):,}")
opt_  = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
sch   = torch.optim.lr_scheduler.CosineAnnealingLR(opt_, T_max=35, eta_min=5e-6)


## STEP 5 · Train (max 35 epochs, patience=8)

In [ ]:
print("\n  STEP 5 — Training (max 35 epochs, patience=8)")
ckpt = os.path.join(OUT,"model.pt")
bvt,bvthr,bad = -2.,.5,0
hist = {"tl":[],"vl":[],"ta":[],"va":[],"vtss":[],"vauc":[]}

for ep in range(1,36):
    tl_,ta_,_,_   = run_ep(model,tr_dl,opt_,ce_,bce_,True)
    vl_,va_,vt,vp = run_ep(model,va_dl,opt_,ce_,bce_,False)
    sch.step()
    vt_thr,vt_tss = bthr(vt,vp)
    try: vauc = roc_auc_score(vt,vp)
    except: vauc = float("nan")
    hist["tl"].append(tl_); hist["vl"].append(vl_)
    hist["ta"].append(ta_); hist["va"].append(va_)
    hist["vtss"].append(vt_tss); hist["vauc"].append(vauc)
    print(f"  Ep{ep:3d}  loss={tl_:.4f}/{vl_:.4f}  acc={ta_:.3f}/{va_:.3f}"
          f"  AUC={vauc:.3f}  TSS@{vt_thr:.2f}={vt_tss:.3f}  ({time.time()-t_all:.0f}s)")
    if vt_tss > bvt:
        bvt,bvthr,bad = vt_tss,vt_thr,0
        torch.save(model.state_dict(), ckpt)
    else:
        bad += 1
        if bad >= 8:
            print(f"  Early stop ep{ep}  best val TSS={bvt:.4f}"); break

model.load_state_dict(torch.load(ckpt, map_location=DEV))
print(f"\n  Best val TSS={bvt:.4f} @ thr={bvthr:.2f}")


## STEP 6 · Evaluate on Synthetic Test Set (C/M/X Detection)

In [ ]:
print("\n  STEP 6 — Synthetic test evaluation")
_, te_acc, te_t, te_p = run_ep(model,te_dl,opt_,ce_,bce_,False)
te_sc = sc_(te_t,te_p,bvthr); _,te_best = bthr(te_t,te_p)
try: rauc = roc_auc_score(te_t,te_p)
except: rauc = float("nan")
fpr,tpr_,_ = roc_curve(te_t,te_p)

model.eval(); preds = []
with torch.no_grad():
    for X,_,__ in te_dl:
        preds.append(model(X.to(DEV))[0].argmax(1).cpu().numpy())
preds = np.concatenate(preds)
f1    = f1_score(yte_n[:len(preds)],preds,average="macro")

print(f"  Nowcast  acc={te_acc:.4f}  macro-F1={f1:.4f}")
print(f"  Forecast AUC={rauc:.4f}  TSS={te_sc['TSS']:.4f}  POD={te_sc['POD']:.4f}  FAR={te_sc['FAR']:.4f}")

# ── Per-class C/M/X detection — synthetic ─────────────────────────────────────
pcm_syn = per_class_metrics(yte_n[:len(preds)], preds, label="Synthetic Test")
pd.DataFrame(pcm_syn).to_csv(os.path.join(OUT,"perclass_metrics_synthetic.csv"),index=False)

# ── Lead time — synthetic ──────────────────────────────────────────────────────
lead_syn = compute_lead_times(flare_db_syn, tte[:len(te_p)], te_p, bvthr, label="Synthetic")
if len(lead_syn): lead_syn.to_csv(os.path.join(OUT,"lead_time_synthetic.csv"),index=False)


## STEP 7 · Evaluate on Real Aditya-L1 Data (C/M/X Detection + Lead Time)

In [ ]:
print("\n  STEP 7 — Real Aditya-L1 evaluation")
Xr,ynr,yfr,ter = build_windows(df_real,WM,SM,FHM)
rds = FDS(Xr,ynr,yfr,mu=tr_ds.mu,sg=tr_ds.sg)
rdl = DataLoader(rds,BS,shuffle=False)
_,re_acc,re_t,re_p = run_ep(model,rdl,opt_,ce_,bce_,False)
re_sc = sc_(re_t,re_p,bvthr); _,re_best = bthr(re_t,re_p)
try: re_auc = roc_auc_score(re_t,re_p)
except: re_auc = float("nan")
fpr_r,tpr_r,_ = roc_curve(re_t,re_p)

re_preds = []
with torch.no_grad():
    for X,_,__ in rdl:
        re_preds.append(model(X.to(DEV))[0].argmax(1).cpu().numpy())
re_preds  = np.concatenate(re_preds)
re_f1     = f1_score(ynr[:len(re_preds)],re_preds,average="macro")

print(f"  Nowcast  acc={re_acc:.4f}  macro-F1={re_f1:.4f}")
print(f"  Forecast AUC={re_auc:.4f}  TSS={re_sc['TSS']:.4f}  POD={re_sc['POD']:.4f}  FAR={re_sc['FAR']:.4f}")

# ── Per-class C/M/X detection — real ─────────────────────────────────────────
pcm_real = per_class_metrics(ynr[:len(re_preds)], re_preds, label="Real Aditya-L1")
pd.DataFrame(pcm_real).to_csv(os.path.join(OUT,"perclass_metrics_real.csv"),index=False)

# ── Lead time — real ──────────────────────────────────────────────────────────
t_wr2    = ter[:len(re_p)]
lead_real = compute_lead_times(flare_db_real, t_wr2, re_p, bvthr, label="Real Aditya-L1")
if len(lead_real): lead_real.to_csv(os.path.join(OUT,"lead_time_real.csv"),index=False)

print(f"\n  Oracle-TSS syn={te_best:.4f}  real={re_best:.4f}")
print(f"  Total elapsed: {time.time()-t_all:.0f}s")


## STEP 8 · Render Visual Alert Dashboard

In [ ]:
print("\n  STEP 8 — Rendering Visual Alert Dashboard")

dash_path = os.path.join(OUT, "visual_dashboard.png")
render_dashboard(
    df_real      = df_real,
    hist         = hist,
    te_t=te_t, te_p=te_p, te_sc=te_sc, rauc=rauc, fpr=fpr, tpr_=tpr_,
    re_t=re_t, re_p=re_p, re_sc=re_sc, re_auc=re_auc, fpr_r=fpr_r, tpr_r=tpr_r,
    preds=preds,    yte_n=yte_n,
    re_preds=re_preds, ynr=ynr,
    bvthr        = bvthr,
    flare_db_real= flare_db_real,
    pcm_syn      = pcm_syn,
    pcm_real     = pcm_real,
    lead_syn     = lead_syn,
    lead_real    = lead_real,
    ter          = ter,
    save_path    = dash_path,
)


## STEP 9 · Save Full Metrics Report

In [ ]:
report = {
    "model"       : "FlareNetV5 (Dual-branch BiLSTM + Residual CNN, 12-channel)",
    "architecture": {"cnn_channels":32,"nowcast_lstm_hidden":32,"forecast_lstm_hidden":40,
                     "dropout":0.35,"input_channels":NCH,"window_min":WM,"stride_min":SM},
    "training"    : {"best_val_TSS":float(bvt),"threshold":float(bvthr),
                     "epochs":len(hist["tl"])},
    "synthetic_test": {
        "accuracy" :float(te_acc), "macro_F1":float(f1),
        "AUC"      :float(rauc),   "TSS":float(te_sc["TSS"]),
        "HSS"      :float(te_sc["HSS"]), "POD":float(te_sc["POD"]),
        "FAR"      :float(te_sc["FAR"]), "TSS_oracle":float(te_best),
        "perclass" : pcm_syn,
    },
    "real_adityal1_test": {
        "accuracy" :float(re_acc), "macro_F1":float(re_f1),
        "AUC"      :float(re_auc), "TSS":float(re_sc["TSS"]),
        "HSS"      :float(re_sc["HSS"]), "POD":float(re_sc["POD"]),
        "FAR"      :float(re_sc["FAR"]), "TSS_oracle":float(re_best),
        "perclass" : pcm_real,
    },
    "flare_db_real_count"    : len(flare_db_real),
    "flare_db_syn_count"     : len(flare_db_syn),
    "flare_db_real"          : flare_db_real.to_dict(orient="records"),
    "lead_time_syn"  : {
        "n_events": len(lead_syn),
        "n_hits"  : int((lead_syn.status=="HIT").sum()) if len(lead_syn)>0 else 0,
        "mean_min": float(lead_syn[lead_syn.status=="HIT"].lead_time_min.mean())
                    if len(lead_syn)>0 and (lead_syn.status=="HIT").any() else None,
    },
    "lead_time_real": {
        "n_events": len(lead_real),
        "n_hits"  : int((lead_real.status=="HIT").sum()) if len(lead_real)>0 else 0,
        "mean_min": float(lead_real[lead_real.status=="HIT"].lead_time_min.mean())
                    if len(lead_real)>0 and (lead_real.status=="HIT").any() else None,
    },
}

with open(os.path.join(OUT,"metrics_report.json"),"w") as f:
    json.dump(report, f, indent=2)

print("\n" + "="*65)
print("  FINAL RESULTS — FlareNet V5")
print("="*65)
print(f"  ── Synthetic Test ──────────────────────────────────────")
print(f"  Nowcast  acc={te_acc:.4f}  macro-F1={f1:.4f}")
print(f"  Forecast AUC={rauc:.4f}  TSS={te_sc['TSS']:.4f}  HSS={te_sc['HSS']:.4f}")
print(f"  POD={te_sc['POD']:.4f}  FAR={te_sc['FAR']:.4f}  Oracle-TSS={te_best:.4f}")
print(f"  ── Real Aditya-L1 ──────────────────────────────────────")
print(f"  Nowcast  acc={re_acc:.4f}  macro-F1={re_f1:.4f}")
print(f"  Forecast AUC={re_auc:.4f}  TSS={re_sc['TSS']:.4f}  HSS={re_sc['HSS']:.4f}")
print(f"  POD={re_sc['POD']:.4f}  FAR={re_sc['FAR']:.4f}  Oracle-TSS={re_best:.4f}")
print(f"  ── Flare DB: syn={len(flare_db_syn)} events  real={len(flare_db_real)} events")
print(f"  ── Outputs → {OUT}/")
print(f"  ── Total time: {time.time()-t_all:.0f}s")
